# 10 — Vector RAG Pipeline: Semantic Fact Retrieval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/10_vector_rag_pipeline.ipynb)

The basic `GroundTruthStore` matches facts by keyword overlap.
`VectorGroundTruthStore` upgrades to **semantic similarity search** —
facts are embedded and retrieved by meaning, not just words.

**What you'll learn:**
1. In-memory vector store (zero dependencies)
2. ChromaDB backend (persistent, scalable)
3. Pluggable backends: Pinecone, Weaviate, Qdrant, FAISS, Elasticsearch
4. Reranking for precision
5. Knowledge base ingestion from documents
6. Multi-tenant fact isolation

In [ ]:
!pip install -q director-ai

## 1. In-Memory Vector Store

The `InMemoryBackend` uses TF-IDF cosine similarity — no external
dependencies, works anywhere. Good for testing and small fact sets.

In [ ]:
from director_ai.core import InMemoryBackend, VectorGroundTruthStore

store = VectorGroundTruthStore(backend=InMemoryBackend())

# Ingest a knowledge base
facts = [
    (
        "pricing_basic",
        "The Basic plan costs $9 per month and includes 1,000 API calls.",
    ),
    (
        "pricing_pro",
        "The Pro plan costs $29 per month with 10,000 API calls and priority support.",
    ),
    ("pricing_enterprise", "Enterprise pricing is custom. Contact sales for a quote."),
    ("sla", "Pro and Enterprise tiers include a 99.9% uptime SLA."),
    ("data_location", "All data is stored in EU (Frankfurt) data centres."),
    ("gdpr", "The platform is fully GDPR compliant with DPA available on request."),
    ("refund", "Refunds are available within 30 days of purchase."),
    (
        "cancellation",
        "Subscriptions can be cancelled at any time. Pro-rata refund applies.",
    ),
]

for key, text in facts:
    store.add(key, text)

print(f"Loaded {len(facts)} facts")

In [ ]:
# Semantic retrieval — "how much" matches pricing even without exact keywords
results = store.retrieve_context_with_chunks("How much does it cost?", top_k=3)

print("Top 3 matches for 'How much does it cost?':")
for chunk in results:
    print(f"  [{chunk.distance:.3f}] {chunk.source}: {chunk.text[:80]}")

In [ ]:
# Score an LLM response against retrieved facts
from director_ai.core import CoherenceScorer

scorer = CoherenceScorer(
    threshold=0.5,
    ground_truth_store=store,
    use_nli=False,
)

# Accurate claim
approved, score = scorer.review(
    "What does the Pro plan include?",
    "The Pro plan is $29/month with 10,000 API calls and priority support.",
)
print(f"Accurate: approved={approved}, coherence={score.score:.3f}")

# Hallucinated claim
approved, score = scorer.review(
    "What does the Pro plan include?",
    "The Pro plan is free and includes unlimited API calls.",
)
print(f"Hallucinated: approved={approved}, coherence={score.score:.3f}")

## 2. ChromaDB Backend (Persistent)

For production, use ChromaDB for persistent storage and real embedding models.

```python
pip install director-ai[vector]
```

```python
from director_ai.core import VectorGroundTruthStore, ChromaBackend

backend = ChromaBackend(
    collection_name="product_knowledge",
    persist_directory="./chroma_data",
)
store = VectorGroundTruthStore(backend=backend)
```

ChromaDB uses `sentence-transformers` embeddings by default.

## 3. Pluggable Backends

Director-AI supports 7 vector backends via optional extras:

| Backend | Install | Use Case |
|---------|---------|----------|
| `InMemoryBackend` | (built-in) | Testing, small fact sets |
| `ChromaBackend` | `[vector]` | Local development, moderate scale |
| Pinecone | `[pinecone]` | Serverless, large scale |
| Weaviate | `[weaviate]` | GraphQL queries, hybrid search |
| Qdrant | `[qdrant]` | High-performance, filtering |
| FAISS | `[faiss]` | CPU-optimised, offline |
| Elasticsearch | `[elasticsearch]` | Existing ES infrastructure |

All implement the `VectorBackend` protocol — swap backends without
changing application code.

## 4. Reranking for Precision

Initial retrieval casts a wide net. A cross-encoder reranker
re-scores candidates for higher precision.

```python
from director_ai.core import RerankedBackend, InMemoryBackend

base = InMemoryBackend()
backend = RerankedBackend(
    base=base,
    reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_k_multiplier=3,       # return top 3 after reranking
)
store = VectorGroundTruthStore(backend=backend)
```

Requires `pip install director-ai[reranker]`.


## 5. Knowledge Base Ingestion

For real applications, ingest documents in bulk.

In [ ]:
import json


def ingest_jsonl(store, filepath: str):
    """Load facts from a JSONL file: {"key": ..., "text": ...} per line."""
    count = 0
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            store.add(row["key"], row["text"])
            count += 1
    return count


def ingest_markdown_sections(store, filepath: str):
    """Split a Markdown file by ## headings, each section becomes a fact."""
    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    sections = content.split("\n## ")
    count = 0
    for section in sections[1:]:  # skip preamble
        lines = section.strip().split("\n", 1)
        heading = lines[0].strip()
        body = lines[1].strip() if len(lines) > 1 else ""
        if body:
            key = heading.lower().replace(" ", "_")[:50]
            store.add(key, f"{heading}: {body[:500]}")
            count += 1
    return count


print("Ingestion helpers defined. Use with any VectorGroundTruthStore.")

## 6. Multi-Tenant Fact Isolation

Each tenant's facts are isolated by a `tenant_id` prefix.
The scorer only retrieves facts matching the active tenant.

In [ ]:
store = VectorGroundTruthStore(backend=InMemoryBackend())

# Tenant A: healthcare
store.add(
    "dosage",
    "Maximum ibuprofen dose: 400mg every 6 hours.",
    tenant_id="clinic_a",
)

# Tenant B: ecommerce
store.add("shipping", "Free shipping on orders over $50.", tenant_id="shop_b")

# Retrieve for tenant A — only sees healthcare facts
ctx_a = store.retrieve_context("What is the max dose?", tenant_id="clinic_a")
print(f"Tenant A context: {ctx_a}")

# Retrieve for tenant B — only sees ecommerce facts
ctx_b = store.retrieve_context("What about shipping?", tenant_id="shop_b")
print(f"Tenant B context: {ctx_b}")

# Cross-tenant isolation: tenant B can't see tenant A's facts
ctx_cross = store.retrieve_context("What is the max dose?", tenant_id="shop_b")
print(f"Cross-tenant (should be empty): '{ctx_cross}'")

## Summary

| Pattern | When to Use |
|---------|-------------|
| `InMemoryBackend` | Tests, <1K facts, no persistence |
| `ChromaBackend` | Local dev, <100K facts, persistent |
| Pinecone/Weaviate/Qdrant | Production, >100K facts, managed |
| FAISS | Offline, CPU-only, bulk search |
| `RerankedBackend` | High precision required |
| `tenant_id` | SaaS, multi-customer deployments |